# OCR Data Manual Correction Validator

This script is used to re-validate the flags after you have manually corrected the OCR data in the CSV files. 
It will:
1. Recalculate `flag_math_total_used`.
2. Recalculate `flag_math_valid_score`.
3. Check for `flag_missing_data`.
4. Set other flags (like name mismatch, linguistic mismatch, OCR timeout) to `False` / `OK` since we assume the manually corrected data is correct and no longer has these issues.

### การตั้งค่า (Configuration)
คุณสามารถระบุเจาะจงได้เลยว่าจะทำกับ `อำเภอ`, `ตำบล`, หรือ `หน่วย` ไหน โดยแก้ตัวแปรด้านล่าง

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
DATA_PATH = "output_data"

# ระบุชื่อ อำเภอ, ตำบล, หรือ หน่วย ที่ต้องการ
# หากต้องการเลือก "ทั้งหมด" ให้ใส่ "*" 
AMPHOE = "อำเภอบ้านไร่"        # ตัวอย่าง: "อำเภอบ้านไร่", "บางบอน", หรือ "*"
TAMBON = "ตำบล" + "คอกควาย"        # ตัวอย่าง: "ตำบลคอกควาย", "บางบอนเหนือ", หรือ "*"
UNIT = "หน่วยเลือกตั้งที่ " + "1"          # ตัวอย่าง: "หน่วยเลือกตั้งที่ 1", หรือ "*"

# สร้าง search pattern ให้ยืดหยุ่นรองรับทุกโครงสร้างโฟลเดอร์
pattern_parts = [DATA_PATH, "**"]
if AMPHOE != "*":
    pattern_parts.append(AMPHOE)
pattern_parts.append(TAMBON)
pattern_parts.append(UNIT)
pattern_parts.append("summary_*.csv")

search_pattern = os.path.join(*pattern_parts)
print(f"Target pattern: {search_pattern}")

In [ ]:
def is_missing(val):
    """Helper to check if a value is considered missing."""
    if pd.isna(val):
        return True
    val_str = str(val).strip()
    return val_str in ('', '-', '—', '.', 'nan', 'NaN', 'None')

def process_csv(file_path):
    df = pd.read_csv(file_path)
    
    for idx, row in df.iterrows():
        # 1. Identify score columns
        score_cols = [col for col in df.columns if col.startswith('scores.')]
        
        # 2. Check for missing data
        ballot_fields = ['valid_ballots', 'invalid_ballots', 'no_vote_ballots', 'ballots_used']
        ballot_fields = [f for f in ballot_fields if f in df.columns]
        
        any_ballot_nan = any(is_missing(row[f]) for f in ballot_fields)
        any_score_nan = any(is_missing(row[sc]) for sc in score_cols)
        
        df.at[idx, 'flag_missing_data'] = any_ballot_nan or any_score_nan
        
        # 3. Calculate flag_math_total_used
        # valid_ballots + invalid_ballots + no_vote_ballots == ballots_used
        if any_ballot_nan:
            df.at[idx, 'flag_math_total_used'] = True
            df.at[idx, 'flag_math_total_used_detail'] = "(Missing Data)"
        else:
            try:
                v = int(float(row['valid_ballots']))
                inv = int(float(row['invalid_ballots']))
                no_v = int(float(row['no_vote_ballots']))
                used = int(float(row['ballots_used']))
                
                expected = v + inv + no_v
                mismatch = expected != used
                
                # 📌 ตรงนี้คือการเซ็ต detail: ถ้าเท่ากันใส่ "OK" ถ้าไม่เท่าใส่ "<expected> != <used>"
                df.at[idx, 'flag_math_total_used'] = mismatch
                df.at[idx, 'flag_math_total_used_detail'] = f"{expected} != {used}" if mismatch else "OK"
            except ValueError:
                df.at[idx, 'flag_math_total_used'] = True
                df.at[idx, 'flag_math_total_used_detail'] = "(Data Format Error)"

        # 4. Calculate flag_math_valid_score
        # sum(scores) == valid_ballots
        valid_ballots_val = row.get('valid_ballots', np.nan)
        valid_score_vals = [row[sc] for sc in score_cols if not is_missing(row[sc])]
        
        if is_missing(valid_ballots_val) or not valid_score_vals:
            df.at[idx, 'flag_math_valid_score'] = True
            df.at[idx, 'flag_math_valid_score_detail'] = "(Missing Data)"
        else:
            try:
                valid_ballots = int(float(valid_ballots_val))
                score_sum = sum(int(float(v)) for v in valid_score_vals)
                
                mismatch = score_sum != valid_ballots
                
                # 📌 ตรงนี้คือการเซ็ต detail: ถ้าเท่ากันใส่ "OK" ถ้าไม่เท่าใส่ "<score_sum> != <valid_ballots>"
                df.at[idx, 'flag_math_valid_score'] = mismatch
                df.at[idx, 'flag_math_valid_score_detail'] = f"{score_sum} != {valid_ballots}" if mismatch else "OK"
            except ValueError:
                df.at[idx, 'flag_math_valid_score'] = True
                df.at[idx, 'flag_math_valid_score_detail'] = "(Data Format Error)"
                
        # 5. Set other flags to False / OK since we assume manually corrected data means no other issues
        df.at[idx, 'flag_name_mismatch'] = False
        df.at[idx, 'flag_name_mismatch_detail'] = "[]"
        df.at[idx, 'flag_linguistic_mismatch'] = False
        df.at[idx, 'flag_linguistic_mismatch_detail'] = "[]"
        df.at[idx, 'flag_ocr_timeout'] = False
        df.at[idx, 'flag_ocr_timeout_detail'] = "OK"
        
    # Save the corrected dataframe back to the same CSV file
    df.to_csv(file_path, index=False)
    print(f"  -> Updated {len(df)} rows in {file_path}")


In [ ]:
# ==========================================
# MAIN EXECUTION
# ==========================================
csv_files = glob.glob(search_pattern, recursive=True)

if not csv_files:
    print(f"❌ No CSV files found matching: '{search_pattern}'")
else:
    print(f"✅ Found {len(csv_files)} files. Starting validation update...")
    for file in csv_files:
        process_csv(file)
    print("\n🎉 All files processed successfully!")
